In [1]:
import os

# Change to a new directory
new_path = '/ocean/projects/mat240020p/nli1/diffusion/nuohao/code-sim/simulate/'
os.chdir(new_path)
!python mask2anno_filter.py

Processing image 0/10000: grid1_roi2_500kx_0p5nm_haadf1_0044_25_rot90.png
Processing image 1/10000: BF X500K, 12_2_rot90.png
Processing image 2/10000: 200kV_500kx_p2nm_8cmCL_grain1_0072 - Copy_22_rot90.png
Processing image 3/10000: BF X500K, 04_2_rot180.png
Processing image 4/10000: 200kV_500kx_p2nm_8cmCL_grain1_0109 - Copy_19_rot90.png
Processing image 5/10000: 200kV_500kx_p2nm_8cmCL_grain1_0012_n_4_original.png
Processing image 6/10000: 2501_300kx_1nm_clhaadf3_0044_7_rot270.png
Processing image 7/10000: K713_300kx_store4_grid1_0009_25_rot270.png
Processing image 8/10000: BF X500K, 01 (2)_23_original.png
Processing image 9/10000: BF X500K, 16_6_rot90.png
Processing image 10/10000: 0501_300kx_1nm_clhaadf3_0050_3_original.png
Processing image 11/10000: grid1_roi1_500kx_0p5nm_haadf1_0003_4_original.png
Processing image 12/10000: 200kV_500kx_p2nm_8cmCL_grain1_0064 - Copy_2_rot90.png
Processing image 13/10000: map5_70kx_14_rot180.png
Processing image 14/10000: 1ROI_100kx_4100CL_foil1_16_ro

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use a non-interactive backend if you don't want to pop up windows
import matplotlib.pyplot as plt

from pycocotools.coco import COCO
from matplotlib.patches import Ellipse

# path = "/Users/nuohaoliu/Documents/data_local/dataset/10/10_rot_flip_10k/all/"
# path = "/Users/nuohaoliu/Documents/data_local/dataset/50/50_exp/"
path = "/ocean/projects/mat240020p/nli1/diffusion/data/dataset2/masks/train/50/"
coco_json = path + "256_crop_6_rot_flip/annotations/train.json"   # Path to your annotation file
simulate_path = path + "synthetic_100/"
output_folder = simulate_path + "images"       # Folder containing the images
output_file = simulate_path + "contours_data.xlsx"
out_sim = simulate_path + "mixed_ellipse_masks"
output_analysis = simulate_path + "distributions"
output_anno = os.path.join(simulate_path,  "annotation")



import os
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from pycocotools.coco import COCO
from matplotlib.patches import Ellipse

os.makedirs(output_folder, exist_ok=True)
os.makedirs(simulate_path, exist_ok=True)
os.makedirs(out_sim, exist_ok=True)
os.makedirs(output_analysis, exist_ok=True)

def plot_mask_and_ellipse_together(coco_json_path, output_dir, num_images_to_show=3):
    """
    1) Loads a COCO annotation file
    2) Decodes the segmentation mask
    3) Draws the mask contour in red
    4) Fits an ellipse to each contour and draws it in blue
    5) Saves each figure to output_dir (no GUI pop-up)

    If your ellipses appear flipped top-to-bottom, keep the 'height - cy' line.
    If they also appear flipped left-to-right, uncomment the 'width - cx' line.
    """



    coco = COCO(coco_json_path)
    image_ids = list(coco.imgs.keys())
    if num_images_to_show is not None:
        image_ids = image_ids[:num_images_to_show]

    for image_id in image_ids:
        img_info = coco.imgs[image_id]
        width = img_info["width"]
        height = img_info["height"]
        file_name = img_info["file_name"]

        fig, ax = plt.subplots(figsize=(5, 5))
        ax.set_title(f"Image ID {image_id} (Contour + Ellipse)")

        # Coordinate system: (0,0) at top-left, x→ right, y→ down
        ax.set_xlim([0, width])
        ax.set_ylim([height, 0])  # so y=0 is at the top, y=height is at the bottom

        ax.set_xticks([])
        ax.set_yticks([])

        ann_ids = coco.getAnnIds(imgIds=image_id)
        anns = coco.loadAnns(ann_ids)

        for ann in anns:
            mask = coco.annToMask(ann)
            if mask.sum() == 0:
                continue

            mask_8u = (mask * 255).astype(np.uint8)
            contours, _ = cv2.findContours(
                mask_8u, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )

            # Draw the mask boundary in red (top-down interpretation)
            ax.contour(mask, levels=[0.5], colors='red', linewidths=1.5, origin='lower')

            # Fit an ellipse for each contour and draw it in blue
            for contour in contours:
                if len(contour) < 5:  # need 5+ points for fitEllipse
                    continue
                (cx, cy), (w, h), angle = cv2.fitEllipse(contour)

                # --- Flip Y if needed (top-down vs bottom-up) ---
                # cy = height - cy  # Convert from top-down to our Matplotlib bottom-up
                # cx = width - cx


                # --- If also flipped left-to-right, do this (uncomment if needed) ---
                # cx = width - cx   # Convert from left-to-right if data is mirrored

                ellipse_patch = Ellipse(
                    xy=(cx, cy),
                    width=w,
                    height=h,
                    angle=angle,
                    fill=False,
                    edgecolor='blue',
                    linewidth=2
                )
                ax.add_patch(ellipse_patch)

        # Save figure (no display)
        base_name = os.path.splitext(file_name)[0]
        out_filename = f"{base_name}_mask_ellipse.png"
        out_path = os.path.join(output_dir, out_filename)
        plt.savefig(out_path, bbox_inches='tight')
        plt.close(fig)


def export_contour_data_to_excel(
    coco_json_path,
    output_excel_path= "contours_data.xlsx"
):
    """
    1. Loads a COCO-style annotation file.
    2. For each annotation, decodes the mask and finds contours with OpenCV.
    3. Computes various properties (area, perimeter, ellipse fit, etc.).
    4. Saves these properties as rows in an Excel file.
    """

    # 1. Load COCO annotations
    coco = COCO(coco_json_path)
    ann_ids = coco.getAnnIds()      # All annotation IDs
    anns = coco.loadAnns(ann_ids)

    # Prepare a list to collect the results
    records = []

    # 2. Iterate over all annotations
    for ann in anns:
        image_id = ann["image_id"]
        ann_id = ann["id"]
        category_id = ann["category_id"]

        # Convert segmentation to binary mask
        mask = coco.annToMask(ann)   # shape: [height, width], 0/1
        if mask.sum() == 0:
            continue

        # Convert to 8-bit for OpenCV
        mask_8u = (mask * 255).astype(np.uint8)

        # 3. Find contours
        contours, _ = cv2.findContours(
            mask_8u,
            mode=cv2.RETR_EXTERNAL,
            method=cv2.CHAIN_APPROX_SIMPLE
        )

        # 4. For each contour, compute geometry
        for c_idx, contour in enumerate(contours):
            # Basic area & perimeter
            contour_area = cv2.contourArea(contour)
            contour_perimeter = cv2.arcLength(contour, closed=True)

            # Prepare ellipse parameters (if possible)
            cx, cy, major_axis, minor_axis, angle = [None]*5

            if len(contour) >= 5:
                # Fit an ellipse
                (center, axes, rotation_angle) = cv2.fitEllipse(contour)
                cx, cy = center         # float x,y
                major_axis, minor_axis = axes
                angle = rotation_angle  # degrees

            # Add a row of data
            record = {
                "image_id": image_id,
                "annotation_id": ann_id,
                "category_id": category_id,
                "contour_index": c_idx,
                "area": contour_area,
                "perimeter": contour_perimeter,
                "ellipse_center_x": cx,
                "ellipse_center_y": cy,
                "ellipse_major_axis": major_axis,
                "ellipse_minor_axis": minor_axis,
                "ellipse_angle_deg": angle
            }
            records.append(record)

    # 5. Convert all records to a DataFrame
    df = pd.DataFrame(records)

    # 6. Export to Excel
    df.to_excel(output_excel_path, index=False, engine='openpyxl')
    print(f"Saved contour data to: {output_excel_path}")


if __name__ == "__main__":


    export_contour_data_to_excel(coco_json, output_excel_path=output_file)


if __name__ == "__main__":
    # Example usage:
    plot_mask_and_ellipse_together(
        coco_json_path=coco_json,
        output_dir=output_folder,
        num_images_to_show=3
    )



loading annotations into memory...
Done (t=1.00s)
creating index...
index created!
Saved contour data to: /ocean/projects/mat240020p/nli1/diffusion/data/dataset2/masks/train/50/synthetic_100/contours_data.xlsx
loading annotations into memory...
Done (t=0.71s)
creating index...
index created!


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# path = "/Users/nuohaoliu/Documents/data_local/dataset/10/10_rot_flip_10k/anno_200kV_500kx_p2nm/"
# coco_json = path + "train.json"   # Path to your annotation file
# output_folder = path + "/images"       # Folder containing the images

# output_file = path + "contours_data.xlsx"
# out_sim = path + "mixed_ellipse_masks"

def plot_distributions_by_category(excel_file, num_bins=30):
    """
    Plot histograms of each parameter by category into separate figures.
    Also plot a histogram of defects-per-image.
    Filters extreme outliers (100x mean rule).
    """

    # Load data
    df = pd.read_excel(excel_file)
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["area", "ellipse_major_axis", "ellipse_minor_axis", "ellipse_angle_deg"])
    df = df[
        (df["area"] > 0) & 
        (df["area"] < 1e9) & 
        (df["ellipse_major_axis"] < 1e6) & 
        (df["ellipse_minor_axis"] < 1e6)
    ]

    # Outlier filter: 100x mean
    params = ["area", "ellipse_major_axis", "ellipse_minor_axis", "ellipse_angle_deg"]
    for param in params:
        mean_val = df[param].mean()
        threshold = 100 * mean_val
        original_len = len(df)
        df = df[df[param] <= threshold]
        removed = original_len - len(df)
        print(f"Filtered out {removed} rows from '{param}' > 100× mean")

    # Plot settings
    category_colors = {1: 'blue', 2: 'orange', 3: 'green'}

    # 👉 Separate plot for each parameter + category
    for param in params:
        if param == "area":
            # 👉 fixed range and bin width for area
            lower = 0
            upper = 6000
            bins = np.arange(lower, upper + 250, 250)
        else:
            # automatic percentile range for others
            lower = df[param].quantile(0.01)
            upper = df[param].quantile(0.99)
            bins = np.linspace(lower, upper, num_bins + 1)

        for cat_id in sorted(category_colors.keys()):
            if cat_id in df["category_id"].unique():
                subset = df[df["category_id"] == cat_id]
                subset = subset[(subset[param] >= lower) & (subset[param] <= upper)]

                fig, ax = plt.subplots()
                #ax.set_title(f"{param.replace('_',' ').title()}\nCategory {cat_id} (N={len(subset)})", fontsize=14)
                ax.hist(
                    subset[param],
                    bins=bins,
                    alpha=0.7,
                    color=category_colors[cat_id],
                    edgecolor='black'
                )
                ax.set_xlabel(param.replace("_", " ").title(), fontsize=12)
                ax.set_ylabel("Frequency", fontsize=12)
                ax.set_xlim(lower, upper)
                if param == "area":
                    if cat_id == 1:
                        ax.set_ylim(0, 3500)
                    elif cat_id == 2:
                        ax.set_ylim(0, 5000)
                    elif cat_id == 3:
                        ax.set_ylim(0, 16000)
                ax.tick_params(axis='both', which='major', labelsize=12)
                # ❌ remove grid
                # ax.grid(True, linestyle='--', alpha=0.3)

                out_fname = output_analysis + f"hist_{param}_category_{cat_id}.png"
                plt.savefig(out_fname, dpi=150, bbox_inches='tight')
                plt.close(fig)

    # 👉 Defects per image plot
    defects_per_image = df.groupby("image_id").size().rename("defects_per_image").reset_index()
    mean_val = defects_per_image["defects_per_image"].mean()
    threshold = 100 * mean_val
    original_len = len(defects_per_image)
    defects_per_image = defects_per_image[defects_per_image["defects_per_image"] <= threshold]
    removed = original_len - len(defects_per_image)
    print(f"Filtered out {removed} image(s) with defects_per_image > 100× mean")

    median_val = defects_per_image["defects_per_image"].median()
    std_val = defects_per_image["defects_per_image"].std()
    min_val = defects_per_image["defects_per_image"].min()
    max_val = defects_per_image["defects_per_image"].max()
    total_images = len(defects_per_image)

    fig, ax = plt.subplots()
    #ax.set_title("Defects per Image", fontsize=14)
    # 👉 fixed range and bin width for defects_per_image
    lower = 0
    upper = 25
    bins = np.arange(lower, upper + 1, 1)

    ax.hist(
        defects_per_image["defects_per_image"],
        bins=bins,
        alpha=0.7,
        edgecolor='black'
    )
    ax.set_xlabel("Defects per Image", fontsize=12)
    ax.set_ylabel("Number of Images", fontsize=12)
    ax.set_xlim(lower, upper)
    ax.set_ylim(0, 1500)
    ax.tick_params(axis='both', which='major', labelsize=12)
    # ❌ remove grid
    # ax.grid(True, linestyle='--', alpha=0.3)

    stats_text = (
        f"Total Images: {total_images}\n"
        f"Mean: {mean_val:.2f}\n"
        f"Median: {median_val:.2f}\n"
        f"Std: {std_val:.2f}\n"
        f"Min: {min_val:.0f}\n"
        f"Max: {max_val:.0f}"
    )
    ax.text(
        0.98, 0.97, stats_text,
        transform=ax.transAxes,
        verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
        fontsize=10
    )

    out_fname = output_analysis + "hist_defects_per_image.png"
    plt.savefig(out_fname, dpi=150, bbox_inches='tight')
    plt.close(fig)

    return df, defects_per_image

if __name__ == "__main__":
    # excel_file = "/Users/nuohaoliu/Documents/data_local/dataset/10/10_rot_flip_10k/anno_200kV_500kx_p2nm"  # Replace with your actual path
    excel_file = output_file
    df, dpi = plot_distributions_by_category(excel_file)

Filtered out 0 rows from 'area' > 100× mean
Filtered out 0 rows from 'ellipse_major_axis' > 100× mean
Filtered out 11 rows from 'ellipse_minor_axis' > 100× mean
Filtered out 0 rows from 'ellipse_angle_deg' > 100× mean
Filtered out 0 image(s) with defects_per_image > 100× mean


In [7]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

sim_num = 10000

def is_bbox_overlap(bbox1, bbox2, buffer=2):
    x1_min, y1_min, w1, h1 = bbox1
    x2_min, y2_min, w2, h2 = bbox2
    x1_max, y1_max = x1_min + w1 + buffer, y1_min + h1 + buffer
    x2_max, y2_max = x2_min + w2 + buffer, y2_min + h2 + buffer
    return not (x1_max < x2_min or x2_max < x1_min or y1_max < y2_min or y2_max < y1_min)

def generate_mixed_ellipse_masks_with_defect_distribution(df, n_masks=1000, out_sim="mixed_ellipse_masks"):
    os.makedirs(out_sim, exist_ok=True)
    skipped = 0
    category_colors = {1: 'blue', 2: 'orange', 3: 'green'}
    defects_per_image_dist = df.groupby("image_id").size()
    cat_distribution = df["category_id"].value_counts(normalize=True).sort_index()
    data_by_cat = {cat_id: df[df["category_id"] == cat_id] for cat_id in cat_distribution.index}
    synthetic_data = []

    for mask_idx in range(n_masks):
        total_ellipses = np.random.choice(defects_per_image_dist)
        fig, ax = plt.subplots(figsize=(2.56, 2.56))
        ax.set_xlim(-128, 128)
        ax.set_ylim(-128, 128)
        ax.set_aspect("equal")
        ax.axis("off")
        existing_bboxes = []

        # Adjust category 1 to appear twice as often
        adjusted_weights = cat_distribution.copy()
        adjusted_weights[1] *= 2
        adjusted_probs = adjusted_weights / adjusted_weights.sum()
        categories = np.random.choice(cat_distribution.index, size=total_ellipses, p=adjusted_probs.values)

        for cat_id in categories:
            row = data_by_cat[cat_id].sample(n=1).iloc[0]
            major = row["ellipse_major_axis"]
            minor = row["ellipse_minor_axis"]
            angle = row["ellipse_angle_deg"]

            placed = False
            for attempt in range(80):
                x = np.random.uniform(-128, 128)
                y = np.random.uniform(-128, 128)
                w = minor
                h = major
                bbox = [x - w / 2, y - h / 2, w, h]
                if all(not is_bbox_overlap(bbox, existing) for existing in existing_bboxes):
                    existing_bboxes.append(bbox)
                    placed = True
                    break

            if not placed:
                print(f"plotted an ellipse after 80 failed attempts to avoid overlap.")
                skipped += 1
                existing_bboxes.append(bbox)

            color = category_colors.get(cat_id, 'black')
            ellipse = Ellipse((x, y), width=minor, height=major, angle=angle,
                              edgecolor=color, facecolor="none", linewidth=1)
            ax.add_patch(ellipse)

            synthetic_data.append({
                "synthetic_mask_id": mask_idx,
                "category_id": cat_id,
                "ellipse_major_axis": major,
                "ellipse_minor_axis": minor,
                "ellipse_angle_deg": angle,
                "center_x": x,
                "center_y": y,
                "area": np.pi * (major/2) * (minor/2)
            })

        fname = os.path.join(out_sim, f"mask_{mask_idx:04d}.png")
        plt.savefig(fname, dpi=100, bbox_inches='tight', pad_inches=0)
        plt.close(fig)

    print(f"Generated {n_masks} synthetic masks in '{out_sim}'.")
    print(f"Skipped {skipped} ellipses in '{out_sim}'.")
    synthetic_data_df = pd.DataFrame(synthetic_data)
    return synthetic_data_df

if __name__ == "__main__":
    
    # Suppose 'df' is your real annotation DataFrame with columns for ellipse geometry
    # Each row is one ellipse. You must have 'image_id' and 'category_id' in df.
    # df = pd.read_excel("/path/to/your/contours_data.xlsx")

    # Generate the synthetic data
    synthetic_df = generate_mixed_ellipse_masks_with_defect_distribution(df, sim_num, output_folder + "/mixed_ellipse_masks")

    # Now synthetic_df has the parameters of all ellipses in the 100 generated images
    synthetic_df.to_csv("synthetic_ellipses.csv", index=False)

plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted 

In [7]:
'''
##sampling simulated masks
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

sim_num = 10000

def is_bbox_overlap(bbox1, bbox2, buffer=2):
    """
    Check whether two bounding boxes overlap.
    bboxes are in format [x_min, y_min, width, height]
    buffer: optional space between ellipses
    """
    x1_min, y1_min, w1, h1 = bbox1
    x2_min, y2_min, w2, h2 = bbox2
    x1_max, y1_max = x1_min + w1 + buffer, y1_min + h1 + buffer
    x2_max, y2_max = x2_min + w2 + buffer, y2_min + h2 + buffer

    return not (x1_max < x2_min or x2_max < x1_min or y1_max < y2_min or y2_max < y1_min)

def generate_mixed_ellipse_masks_with_defect_distribution(
    df, 
    n_masks=1000, 
    out_sim="mixed_ellipse_masks"
):
    """
    Generates synthetic ellipse masks using the real distribution of:
      1) defects per image (from df), 
      2) category proportions, 
      3) ellipse major/minor/angle.

    Args:
        df:         DataFrame with columns at least ["image_id","category_id",
                    "ellipse_major_axis","ellipse_minor_axis","ellipse_angle_deg"].
                    Each row = one ellipse annotation in real data.
        n_masks:    Number of synthetic images to generate.
        out_sim:    Output folder path to save images.

    Returns:
        synthetic_data_df: DataFrame of the synthetic ellipse parameters.
    """
    os.makedirs(out_sim, exist_ok=True)
    skipped = 0

    # Define category -> color mapping
    category_colors = {1: 'blue', 2: 'orange', 3: 'green'} #plot color

    # 1. Create a distribution for "defects per image" from real data
    defects_per_image_dist = df.groupby("image_id").size()

    # 2. (Optional) Category distribution
    cat_distribution = df["category_id"].value_counts(normalize=True).sort_index()

    # 3. For convenience, group real data by category
    data_by_cat = {
        cat_id: df[df["category_id"] == cat_id]
        for cat_id in cat_distribution.index
    }

    # 4. Prepare to store synthetic data
    synthetic_data = []
    

    for mask_idx in range(n_masks):
        # Decide how many ellipses this synthetic mask will have
        total_ellipses = np.random.choice(defects_per_image_dist)

        # Create figure
        fig, ax = plt.subplots(figsize=(2.56, 2.56))  # ~256x256 px if dpi=100
        ax.set_xlim(-128, 128)
        ax.set_ylim(-128, 128)
        ax.set_aspect("equal")
        ax.axis("off")  # no axis for the synthetic masks

        existing_bboxes = []

        for _ in range(total_ellipses):
            # Pick a category
            cat_id = np.random.choice(cat_distribution.index, p=cat_distribution.values)
            row = data_by_cat[cat_id].sample(n=1).iloc[0]
            major = row["ellipse_major_axis"]
            minor = row["ellipse_minor_axis"]
            angle = row["ellipse_angle_deg"]

            # Try up to N times to place this ellipse without overlap
            placed = False
            for attempt in range(80):  # Max attempts to find non-overlapping placement
                x = np.random.uniform(-128, 128)
                y = np.random.uniform(-128, 128)

                # Convert ellipse to bounding box (centered)
                w = minor
                h = major
                bbox = [x - w / 2, y - h / 2, w, h]

                # Check for overlap
                if all(not is_bbox_overlap(bbox, existing) for existing in existing_bboxes):
                    existing_bboxes.append(bbox)  # Keep track of this one
                    placed = True
                    break  # Valid placement found

            if not placed:
                print(f"plotted an ellipse after 80 failed attempts to avoid overlap.")
                skipped += 1
                existing_bboxes.append(bbox)
                #continue  # Skip this ellipse

            # Plot it
            color = category_colors.get(cat_id, 'black')
            ellipse = Ellipse((x, y), width=minor, height=major, angle=angle,
                            edgecolor=color, facecolor="none", linewidth=1)
            ax.add_patch(ellipse)

            # Store synthetic info
            synthetic_data.append({
                "synthetic_mask_id": mask_idx,
                "category_id": cat_id,
                "ellipse_major_axis": major,
                "ellipse_minor_axis": minor,
                "ellipse_angle_deg": angle,
                "center_x": x,
                "center_y": y,
                "area": np.pi * (major/2) * (minor/2)
            })

        # 5. Save mask image
        fname = os.path.join(out_sim, f"mask_{mask_idx:04d}.png")
        plt.savefig(fname, dpi=100, bbox_inches='tight', pad_inches=0)
        plt.close(fig)

    print(f"Generated {n_masks} synthetic masks in '{out_sim}'.")
    print(f"skipped {skipped} ellipses in '{out_sim}'.")

    # Convert synthetic_data to a DataFrame for your records
    synthetic_data_df = pd.DataFrame(synthetic_data)
    return synthetic_data_df

# EXAMPLE USAGE:
if __name__ == "__main__":
    
    # Suppose 'df' is your real annotation DataFrame with columns for ellipse geometry
    # Each row is one ellipse. You must have 'image_id' and 'category_id' in df.
    # df = pd.read_excel("/path/to/your/contours_data.xlsx")

    # Generate the synthetic data
    synthetic_df = generate_mixed_ellipse_masks_with_defect_distribution(df, sim_num, output_folder + "/mixed_ellipse_masks")

    # Now synthetic_df has the parameters of all ellipses in the 100 generated images
    synthetic_df.to_csv("synthetic_ellipses.csv", index=False)
'''

plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted an ellipse after 80 failed attempts to avoid overlap.
plotted 

In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
output_dis = simulate_path + "syn_dis"

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_synthetic_distributions(synth_df, num_bins=30, out_dir=output_dis):
    """
    Plots:
      1) Histograms of area, ellipse_major_axis, ellipse_minor_axis, ellipse_angle_deg
         for each category_id.
      2) A separate histogram of 'defects per synthetic mask' (counts how many ellipses
         each synthetic_mask_id contains).
    Saves all plots in out_dir.
    """

    os.makedirs(out_dir, exist_ok=True)

    params = ["area", "ellipse_major_axis", "ellipse_minor_axis", "ellipse_angle_deg"]
    category_colors = {1: "blue", 2: "orange", 3: "green"}

    #
    # 1) Plot distributions of the four ellipse parameters by category
    #
    for param in params:
        if param == "area":
            # 👉 fixed range for area
            lower = 0
            upper = 6000
            bins = np.arange(lower, upper + 250, 250)
        else:
            # automatic percentile range for others
            lower = synth_df[param].quantile(0.01)
            upper = synth_df[param].quantile(0.99)
            bins = np.linspace(lower, upper, num_bins + 1)

        for cat_id in sorted(category_colors.keys()):
            if cat_id in synth_df["category_id"].unique():
                subset = synth_df[synth_df["category_id"] == cat_id]
                subset = subset[(subset[param] >= lower) & (subset[param] <= upper)]

                fig, ax = plt.subplots()
                #ax.set_title(f"{param.replace('_',' ').title()}\nCategory {cat_id} (N={len(subset)})", fontsize=14)
                ax.hist(
                    subset[param],
                    bins=bins,
                    alpha=0.7,
                    color=category_colors[cat_id],
                    edgecolor='black'
                )
                ax.set_xlabel(param.replace("_", " ").title(), fontsize=12)
                ax.set_ylabel("Frequency", fontsize=12)
                ax.set_xlim(lower, upper)
                if param == "area":
                    if cat_id == 1:
                        ax.set_ylim(0, 6500)
                    elif cat_id == 2:
                        ax.set_ylim(0, 5000)
                    elif cat_id == 3:
                        ax.set_ylim(0, 16000)
                ax.tick_params(axis='both', which='major', labelsize=12)
                # ❌ no grid
                # ax.grid(True, linestyle="--", alpha=0.3)

                out_path = os.path.join(out_dir, f"synthetic_hist_{param}_category_{cat_id}.png")
                plt.savefig(out_path, dpi=150, bbox_inches="tight")
                plt.close(fig)

    #
    # 2) Plot distribution of "defects per synthetic mask"
    #
    defects_per_mask = synth_df.groupby("synthetic_mask_id").size().rename("defects_per_mask")
    desc = defects_per_mask.describe()
    total_masks = int(desc["count"])
    mean_val = desc["mean"]
    median_val = defects_per_mask.median()
    std_val = desc["std"]
    min_val = desc["min"]
    max_val = desc["max"]

    fig, ax = plt.subplots()
    #ax.set_title("Defects per Synthetic Mask", fontsize=14)

    # 👉 fixed range and bin width
    lower = 0
    upper = 25
    bins = np.arange(lower, upper + 1, 1)

    ax.hist(
        defects_per_mask,
        bins=bins,
        alpha=0.7,
        edgecolor="black"
    )
    ax.set_xlabel("Defects per Mask", fontsize=12)
    ax.set_ylabel("Number of Masks", fontsize=12)
    ax.set_xlim(lower, upper)
    ax.set_ylim(0, 1500)
    ax.tick_params(axis='both', which='major', labelsize=12)
    # ❌ no grid
    # ax.grid(True, linestyle="--", alpha=0.3)

    stats_text = (
        f"Total Masks: {total_masks}\n"
        f"Mean: {mean_val:.2f}\n"
        f"Median: {median_val:.2f}\n"
        f"Std: {std_val:.2f}\n"
        f"Min: {int(min_val)}\n"
        f"Max: {int(max_val)}"
    )
    ax.text(
        0.98, 0.97,
        stats_text,
        transform=ax.transAxes,
        verticalalignment="top",
        horizontalalignment="right",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
        fontsize=10
    )

    out_path = os.path.join(out_dir, "synthetic_hist_defects_per_mask.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Synthetic parameter distribution plots saved in: {out_dir}/")


# EXAMPLE USAGE (assuming you already have synthetic_df):
synthetic_df = pd.read_csv("synthetic_ellipses.csv")
plot_synthetic_distributions(synthetic_df)


Synthetic parameter distribution plots saved in: /ocean/projects/mat240020p/nli1/diffusion/data/dataset2/masks/train/50/synthetic_100/syn_dis/


In [10]:



import os
import numpy as np
import pandas as pd
import cv2
from PIL import Image

output_dir = out_sim 
def create_rgb_masks_from_synthetic_df_cv(synth_df, output_dir="rgb_masks", image_size=256):
    os.makedirs(output_dir, exist_ok=True)

    # Define category colors: RGB tuples
    category_colors = {
        1: (255, 0, 0),     # Blue
        2: (0, 255, 0),     # Green
        3: (0, 0, 255)     # Red
    }

    grouped = synth_df.groupby("synthetic_mask_id")

    for mask_id, group in grouped:
        canvas = np.zeros((image_size, image_size, 3), dtype=np.uint8)  # black background

        for _, row in group.iterrows():
            cat_id = int(row["category_id"])
            major = float(row["ellipse_major_axis"])
            minor = float(row["ellipse_minor_axis"])
            angle = float(row["ellipse_angle_deg"])
            cx = float(row["center_x"]) + image_size // 2
            cy = float(row["center_y"]) + image_size // 2

            center = (int(round(cx)), int(round(cy)))
            axes = (int(round(minor / 2)), int(round(major / 2)))  # (minor, major) in OpenCV
            angle = -angle  # OpenCV rotates clockwise

            color = category_colors[cat_id]

            # Draw filled ellipse without anti-aliasing
            cv2.ellipse(canvas, center, axes, angle, 0, 360, color, thickness=-1, lineType=cv2.LINE_8)

        out_path = os.path.join(output_dir, f"00{mask_id:04d}.png")
        Image.fromarray(canvas).save(out_path)

    print(f"Saved clean RGB masks (no gradients) to: {output_dir}/")


create_rgb_masks_from_synthetic_df_cv(synthetic_df, output_dir, image_size=256)

Saved clean RGB masks (no gradients) to: /ocean/projects/mat240020p/nli1/diffusion/data/dataset2/masks/train/50/synthetic_100/mixed_ellipse_masks/


In [11]:
#######3 channel to 1 channel:
# Channel activated	Grayscale value assigned	Meaning
# Red (>80)	    85	 Class 1   100loop
# Green (>80)	170	 Class 2   111loop
# Blue (>80)	255	 Class 3   blackdot
import torch

import os

import numpy as np

from PIL import Image

import cv2
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
def reverse_image_processing(input_folder, output_folder):

    os.makedirs(output_folder, exist_ok=True)
 
    values = torch.tensor([int(255 / 3), 2 * int(255 / 3), 255], dtype=torch.uint8).to(device)
    # => [85, 170, 255]
 
    for filename in os.listdir(input_folder):

        if filename.endswith((".png", ".jpg", ".tif")):

            input_path = os.path.join(input_folder, filename)
 
            img = Image.open(input_path)

            img_array = torch.tensor(np.array(img), dtype=torch.uint8).to(device)
 
            combined = torch.zeros((img_array.shape[0], img_array.shape[1]), dtype=torch.uint8, device=device)
 
            for i, value in enumerate(values):

                combined[img_array[:, :, i] > 80] = value
 
            rgb_combined = torch.stack([combined] * 3, dim=-1)

            output_img = Image.fromarray(rgb_combined.cpu().numpy(), mode="RGB")

            output_path = os.path.join(output_folder, filename)

            output_img.save(output_path)
 
            print(f"Processed and saved: {output_path}")
 
input_folder = output_dir  # Replace with your input folder path

output_folder = simulate_path + "1c_mask"  # Replace with your output folder path
os.makedirs(output_folder, exist_ok=True)

ranges = [(50, 100), (130, 180), (220, 255)]

reverse_image_processing(input_folder, output_folder)
 

Processed and saved: /ocean/projects/mat240020p/nli1/diffusion/data/dataset2/masks/train/50/synthetic_100/1c_mask/004933.png
Processed and saved: /ocean/projects/mat240020p/nli1/diffusion/data/dataset2/masks/train/50/synthetic_100/1c_mask/007224.png
Processed and saved: /ocean/projects/mat240020p/nli1/diffusion/data/dataset2/masks/train/50/synthetic_100/1c_mask/004501.png
Processed and saved: /ocean/projects/mat240020p/nli1/diffusion/data/dataset2/masks/train/50/synthetic_100/1c_mask/005069.png
Processed and saved: /ocean/projects/mat240020p/nli1/diffusion/data/dataset2/masks/train/50/synthetic_100/1c_mask/001390.png
Processed and saved: /ocean/projects/mat240020p/nli1/diffusion/data/dataset2/masks/train/50/synthetic_100/1c_mask/001602.png
Processed and saved: /ocean/projects/mat240020p/nli1/diffusion/data/dataset2/masks/train/50/synthetic_100/1c_mask/001974.png
Processed and saved: /ocean/projects/mat240020p/nli1/diffusion/data/dataset2/masks/train/50/synthetic_100/1c_mask/002861.png


In [1]:
####### new annotations from simulate masks

import json
from PIL import Image, ImageDraw, ImagePath
import os
import numpy as np
import cv2
import shutil
import pandas as pd
from pathlib import Path
import random

# Parameters
option = 'train'  # 'train' or 'val'
imganno_save = False
size = 256
area_threshold = 0
sample_count =10

skip_image = 0
success_count = 0

# Set paths
# current_dir = Path.cwd()

# parent_directory = simulate_path
# parent_directory = Path('/Users/nuohaoliu/Documents/data_local/val/tiny')
path='/ocean/projects/mat240020p/nli1/diffusion/nuohao/maskrcnn/50/mix_100/'
path = Path(path)  # Make sure `path` is a Path object
simulate_path = path #/ "epoch_89"

parent_directory = simulate_path
mask_directory = parent_directory

# Ensure necessary directories exist
create_folder = [
    parent_directory / 'annotations',
    # parent_directory /  f'{option}',
    
]
for folder in create_folder:
    folder_path = Path(folder)
    folder_path.mkdir(parents=False, exist_ok=True)

# annotation_path = parent_directory / 'annotations' / 'val.json'
if imganno_save:
    image_directory = parent_directory / 'train'
    image_directory / f'image_anno_f{area_threshold}'
    create_folder = [
        # parent_directory / 'annotations',
        parent_directory /  f'{option}',
        
    ]
    for folder in create_folder:
        folder_path = Path(folder)
        folder_path.mkdir(parents=False, exist_ok=True)
    
    imageanno_directory = image_directory  / f'image_anno_f{area_threshold}'
# output_directory = parent_directory / 'crop' / f'{option}'
output_annotation_path = parent_directory /  'annotations' / f'{option}.json'

# output_directory.mkdir(parents=True, exist_ok=True)



 
train_dict = dict()
train_dict["licenses"] = [
    {
      "id": 1,
      "name": "",
      "url": ""
    }
  ]
train_dict["categories"]= [
    {
      "id": 1,
      "name": "np",
      "supercategory": "np"
    }
  ]
train_dict["info"]= {
    "contributor": "",
    "date_created": "",
    "description": "New dataset",
    "url": "",
    "version": "1.0",
    "year": 2024
  },
train_dict["images"] = list()
train_dict["annotations"] = list()
 
defect_count = 0
image_count = 0
# gray_levels = [85, 170, 255]
# Define gray levels and their ranges
gray_level_ranges = {
    85: (75, 96),  # Example: 85 ± 5
    170: (160, 180),  # Example: 170 ± 5
    255: (245, 270)  # Example: 255 ± 5
}


# Define a list of valid image extensions
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

# Get the list of image names in the directory with valid extensions
train_img_list = [
    image_name for image_name in os.listdir(mask_directory)
    if image_name.lower().endswith(valid_extensions)
]

def process_mask(im_name):
    # Load the mask and original image
    im_arr = cv2.imread(str(mask_directory / im_name), cv2.IMREAD_GRAYSCALE)

    y_size_mask = im_arr.shape[0]
    x_size_mask = im_arr.shape[1]
    # print('Mask size', x_size_mask, y_size_mask)

    # Check if size matches the required dimensions
    if not y_size_mask == x_size_mask == size:
        raise ValueError("Mask size doesn't match expected size")

    # Prepare for multiple class categories based on gray levels
    # gray_levels = [85, 170, 255]  # Example gray levels for different classes
    class_annotations = {level: {"bboxes": [], "polygons": []} for level in gray_level_ranges}

    # Set the area threshold
    # area_threshold = 500  # Change this value as needed

    for gray_level, (lower_bound, upper_bound) in gray_level_ranges.items():
        # Isolate mask region for the current class within the range
        class_mask = cv2.inRange(im_arr, lower_bound, upper_bound)
        # Isolate mask region for the current class
        # class_mask = (im_arr == gray_level).astype(np.uint8) * 255

        # Get contours for the current class
        contours, _ = cv2.findContours(class_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

        # Filter out small contours based on area
        filtered_contours = [cnt for cnt in contours if cv2.contourArea(cnt) >= area_threshold]

        # Extract polygons and bounding boxes for the current class
        for obj in filtered_contours:
            coords = []
            for point in obj:
                coords.append(int(point[0][0]))
                coords.append(int(point[0][1]))

            # Only append polygons that have 3 or more points
            if len(coords) >= 6:
                # Add the polygon to the current class
                class_annotations[gray_level]["polygons"].append(coords)

                # Calculate bounding box
                xs = coords[::2]  # Even indices are x-coordinates
                ys = coords[1::2]  # Odd indices are y-coordinates
                minx, maxx = min(xs), max(xs)
                miny, maxy = min(ys), max(ys)
                class_annotations[gray_level]["bboxes"].append([minx, miny, maxx, maxy])

    return class_annotations, x_size_mask, y_size_mask

def process_image(im_name):
    # Load the mask and original image
    im_arr = cv2.imread(str(mask_directory / im_name), cv2.IMREAD_GRAYSCALE)
    im_raw_arr = cv2.imread(str(image_directory / im_name))
    im_raw = Image.fromarray(im_raw_arr)

    # Get shape of mask and original image
    y_size = im_raw_arr.shape[0]
    x_size = im_raw_arr.shape[1]
    print('Image size', x_size, y_size)
    y_size_mask = im_arr.shape[0]
    x_size_mask = im_arr.shape[1]
    print('Mask size', x_size_mask, y_size_mask)

    # Resize the mask to match the original image size
    im_arr = cv2.resize(im_arr, (x_size, y_size))

    print('Resized', im_arr.shape[1], im_arr.shape[0])

    # Prepare for multiple class categories based on gray levels
    gray_levels = {
    85: (80, 90),  # Example: 85 ± 5
    170: (165, 175),  # Example: 170 ± 5
    255: (250, 260)  # Example: 255 ± 5
}
    class_annotations = {level: {"bboxes": [], "polygons": []} for level in gray_levels}
    
    # Set the area threshold
    # area_threshold = 500  # Change this value as needed

    for gray_level, (lower_bound, upper_bound) in gray_level_ranges.items():
        # Isolate mask region for the current class within the range
        class_mask = cv2.inRange(im_arr, lower_bound, upper_bound)

        # Get contours for the current class
        contours, _ = cv2.findContours(class_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

        # Filter out small contours based on area
        filtered_contours = [cnt for cnt in contours if cv2.contourArea(cnt) >= area_threshold]

        # Extract polygons and bounding boxes for the current class
        for obj in filtered_contours:
            coords = []
            for point in obj:
                coords.append(int(point[0][0]))
                coords.append(int(point[0][1]))

            # Only append polygons that have 3 or more points
            if len(coords) >= 6:
                # Add the polygon to the current class
                class_annotations[gray_level]["polygons"].append(coords)

                # Calculate bounding box
                xs = coords[::2]  # Even indices are x-coordinates
                ys = coords[1::2]  # Odd indices are y-coordinates
                minx, maxx = min(xs), max(xs)
                miny, maxy = min(ys), max(ys)
                class_annotations[gray_level]["bboxes"].append([minx, miny, maxx, maxy])


    # Plot annotations on the image for visualization
    im1 = ImageDraw.Draw(im_raw)
    for gray_level, data in class_annotations.items():
        color = "blue" if gray_level == 85 else "green" if gray_level == 170 else "red"
        for polygon in data["polygons"]:
            im1.polygon(polygon, outline=color)

        for bbox in data["bboxes"]:
            im1.rectangle(((bbox[0], bbox[1]), (bbox[2], bbox[3])), outline=color)

    return class_annotations, im_raw, x_size, y_size


# Define the fraction or count of images to sample
  # Adjust to your preference
sampled_images = random.sample(train_img_list, int(min(len(train_img_list), sample_count)))


# Choose which set of images to iterate over
if imganno_save:
    # 1) Loop over ALL images to add minimal info to train_dict
#    (e.g., store mask info or at least a placeholder).
#    You can call `process_mask(im_name)` if you want to store the mask polygons for each image.

    image_count = 0
    for i, im_name in enumerate(train_img_list, start=1):
        try:
            # If you want to extract basic mask info for *all* images, call `process_mask`:
            
            class_annotations, x_size, y_size = process_mask(im_name)
            # print('Processing image:', im_name)
            print(f"Processing image {i}/{len(train_img_list)}: {im_name}")
            # You can store the mask-based bboxes/polygons in a top-level data structure
            # or simply store the image metadata here. 
            # For instance, add an 'images' entry to train_dict:
            d = {
                "coco_url": "",
                "date_captured": "",
                "file_name": im_name,
                "flickr_url": "",
                "height": y_size,
                "id": image_count,
                "license": 1,
                "width": x_size
            }
            train_dict['images'].append(d)

            # Append annotation data for only the sampled images
            for gray_level, data in class_annotations.items():
                for bbox, polygon in zip(data["bboxes"], data["polygons"]):
                    bbox_mask = [
                        bbox[0], 
                        bbox[1], 
                        bbox[2] - bbox[0],  # width
                        bbox[3] - bbox[1]   # height
                    ]

                    # Map the range-based gray_level to category_id
                    category_id = None
                    for idx, (level, (lower_bound, upper_bound)) in enumerate(gray_level_ranges.items()):
                        if lower_bound <= gray_level <= upper_bound:
                            category_id = idx + 1  # Category IDs are 1-based
                            break
                    if category_id is None:
                        # If no category_id found, skip
                        continue

                    d = {
                        "image_id": image_count,
                        "id": defect_count,
                        "category_id": category_id,
                        "segmentation": [polygon],
                        "bbox": bbox_mask,
                        "area": bbox_mask[2] * bbox_mask[3],
                        "iscrowd": 0,
                    }
                    train_dict['annotations'].append(d)
                    defect_count += 1
            
            # Optionally, you can store preliminary polygons here if you want
            # to keep "mask info" for all images. For example:
            # (But if you only want to do annotation for sampled images, you might skip writing them here.)
            
            # for gray_level, data in class_annotations.items():
            #    # Store or do something with data['bboxes'] / data['polygons']
            #    # (No final annotation yet, unless you actually want partial annotation.)
            
            image_count += 1

        except Exception as e:
            print(f"Error processing mask for {im_name}: {e}")
            skip_image += 1
            continue

    print(f"Added base info for {image_count} images into train_dict")
    
    # 2) Now handle SAMPLED images for full annotation & saving
    #    This is where we call `process_image(...)` and store polygons, bounding boxes, etc.

    # If you still need the dictionary that maps a file name -> image ID (from the first step),
    # create a quick lookup. For example:
    filename_to_imgid = {}
    for img_info in train_dict['images']:
        filename_to_imgid[img_info["file_name"]] = img_info["id"]

    # Loop over only the sampled subset
    for im_name in sampled_images:
        print('Processing annotation for sampled image:', im_name)
        try:
            class_annotations, im_raw, x_size, y_size = process_image(im_name)
            success_count += 1
        except Exception as e:
            print(f"Error processing image {im_name}: {e}")
            skip_image += 1
            continue

        # Save the annotated image
        im_raw.save(imageanno_directory / im_name)
        # print("saved annotated image:", im_name, x_size, y_size)

        # Retrieve the 'id' from step (1). If you didn't store it, you can re-assign:
        image_id = filename_to_imgid.get(im_name)
        if image_id is None:
            # If for some reason it wasn't added in step (1), handle gracefully
            # e.g., skip or create a new entry
            continue

    print(f"Annotated and saved {success_count} sampled images, skipped {skip_image} images")


        
else:
    for i, im_name in enumerate(train_img_list):
        try:
            class_annotations,x_size, y_size = process_mask(im_name)
            success_count += 1
            print(f"Processing image {i}/{len(train_img_list)}: {im_name}")
        except Exception as e:
            print(f"Error processing image {im_name}: {e}")
            # Optionally handle the error, e.g., skip this image or log the issue
            skip_image += 1
            continue
        
        
    
    
    # shutil.copy(image_directory + im_name, save_im+'/')
 
    # # Save data for YOLO
    # with open(save_anno+'/'+im_name[0:-5]+'.txt', 'w') as f:
    #     for bbox_frac in bboxes_frac:
    #         f.write('0 '+str(bbox_frac[0])+' '+str(bbox_frac[1])+' '+str(bbox_frac[2])+' '+str(bbox_frac[3])+'\n')
 
    # Append data for CoCo (Mask)

        d =  {"coco_url": "",
        "date_captured": "",
        "file_name": im_name,
        "flickr_url": "",
        "height": y_size,
        "id": image_count,
        "license": 1,
        "width": x_size}
        train_dict['images'].append(d)
    
        for gray_level, data in class_annotations.items():
                for bbox, polygon in zip(data["bboxes"], data["polygons"]):
                    bbox_mask = [bbox[0], bbox[1], bbox[2] - bbox[0], bbox[3] - bbox[1]]
                    
                    category_id = None
                    for idx, (level, (lower_bound, upper_bound)) in enumerate(gray_level_ranges.items()):
                        if lower_bound <= gray_level <= upper_bound:
                            category_id = idx + 1  # Category IDs are 1-based
                            break
                    if category_id is None:
                        continue
                    d = {
                        "image_id": image_count,
                        "id": defect_count,
                        "category_id": category_id,  # Map gray level to category_id
                        "segmentation": [polygon],
                        "bbox": bbox_mask,
                        "area": bbox_mask[2] * bbox_mask[3],
                        "iscrowd": 0,
                    }
                    train_dict['annotations'].append(d)
                    defect_count += 1
    
        image_count += 1
 
with open(output_annotation_path, 'w') as f:
    json.dump(train_dict, f, indent=4)
print(f"process {image_count} images, skip {skip_image} images")
print(f"Final train_dict has {len(train_dict['images'])} images, "
      f"{len(train_dict['annotations'])} annotations")

Processing image 0/20800: 004933.png
Processing image 1/20800: 007224.png
Processing image 2/20800: grid1_roi2_500kx_0p5nm_haadf1_0047_12_original.png
Processing image 3/20800: grid1_roi1_500kx_0p5nm_haadf1_0027_28_flip_h.png
Processing image 4/20800: 004501.png
Processing image 5/20800: 005069.png
Processing image 6/20800: 001390.png
Processing image 7/20800: 001602.png
Processing image 8/20800: 001974.png
Processing image 9/20800: K713_300kx_store4_grid1_0007_33_original.png
Processing image 10/20800: 002861.png
Processing image 11/20800: 009299.png
Processing image 12/20800: grid1_roi1_500kx_0p5nm_haadf1_0027_4_flip_h.png
Processing image 13/20800: 007536.png
Processing image 14/20800: 000185.png
Processing image 15/20800: 000783.png
Processing image 16/20800: 006046.png
Processing image 17/20800: grid1_roi2_500kx_0p5nm_haadf1_0053_9_rot180.png
Processing image 18/20800: 004962.png
Processing image 19/20800: 001968.png
Processing image 20/20800: 002204.png
Processing image 21/20800:

In [8]:
from PIL import Image
import numpy as np
import os

# Choose your image path
output_folder = "/Users/nuohaoliu/Documents/data_local/dataset/10/10_rot_flip_10k/all/synthetic/1c_mask"
img_path = os.path.join(output_folder , "000000.png")  # change name/path if needed

# img_path = os.path.join(output_dir, "2ROI_100kx_4100CL_foil1_1_flip_v_rot90.png")  # change name/path if needed
# Load the image
img = Image.open(img_path).convert("RGB")
img_np = np.array(img)

# Print shape info
print(f"Image shape: {img_np.shape} (Height, Width, Channels)")

# Optional: check unique color values in the image
unique_colors = np.unique(img_np.reshape(-1, 3), axis=0)
print("Unique RGB values in image:")
print(unique_colors)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/nuohaoliu/Documents/data_local/dataset/10/10_rot_flip_10k/all/synthetic/1c_mask/000000.png'

In [ ]:
list(synthetic_df.columns)


['image_id',
 'category_id',
 'ellipse_major_axis',
 'ellipse_minor_axis',
 'ellipse_angle_deg',
 'ellipse_center_x',
 'ellipse_center_y',
 'area']

In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd


def ellipse_to_polygon(cx, cy, major, minor, angle_deg, num_points=36):
    theta_rad = math.radians(angle_deg)
    a = major / 2.0
    b = minor / 2.0
    coords = []
    for i in range(num_points):
        t = 2.0 * math.pi * i / num_points
        xp = a * math.cos(t)
        yp = b * math.sin(t)
        x_rot = xp * math.cos(theta_rad) - yp * math.sin(theta_rad)
        y_rot = xp * math.sin(theta_rad) + yp * math.cos(theta_rad)
        coords.append((cx + x_rot, cy + y_rot))
    return coords


def polygon_to_bbox(polygon):
    xs = [p[0] for p in polygon]
    ys = [p[1] for p in polygon]
    return [min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)]


def polygon_area(points):
    area = 0.0
    n = len(points)
    for i in range(n):
        j = (i + 1) % n
        area += points[i][0] * points[j][1] - points[j][0] * points[i][1]
    return abs(area) / 2.0


def synthetic_df_to_coco_json(synth_df, output_path, image_width=256, image_height=256):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    coco = {
        "licenses": [
            {
                "id": 1,
                "name": "",
                "url": ""
            }
        ],
        "info": {
            "contributor": "Nuohao Liu",
            "date_created": "",
            "description": "Simulate ellipses",
            "url": "",
            "version": "1.0",
            "year": 2025
        },
        "categories": [
            {
                "id": 1,
                "name": "np",
                "supercategory": "np"
            }
        ],
        "images": [],
        "annotations": []
    }

    image_ids = sorted(synth_df["image_id"].unique())
    for img_id in image_ids:
        coco["images"].append({
            "coco_url": "",
            "date_captured": "",
            "file_name": f"mask_{img_id:04d}.png",
            "flickr_url": "",
            "height": image_height,
            "width": image_width,
            "id": int(img_id),
            "license": 1
        })

    ann_id_counter = 0
    for _, row in synth_df.iterrows():
        mask_id = int(row["image_id"])
        cat_id = 1  # Use fixed category ID '1' for 'np'
        major = float(row["ellipse_major_axis"])
        minor = float(row["ellipse_minor_axis"])
        angle = float(row["ellipse_angle_deg"])
        cx = float(row["ellipse_center_x"])
        cy = float(row["ellipse_center_y"])

        polygon = ellipse_to_polygon(cx, cy, major, minor, angle)
        polygon_int = [(round(x), round(y)) for (x, y) in polygon]
        segmentation = [coord for xy in polygon_int for coord in xy]
        bbox = [round(v) for v in polygon_to_bbox(polygon_int)]
        area = polygon_area(polygon_int)

        coco["annotations"].append({
            "id": ann_id_counter,
            "image_id": mask_id,
            "category_id": cat_id,
            "segmentation": [segmentation],
            "bbox": bbox,
            "area": area,
            "iscrowd": 0
        })
        ann_id_counter += 1

    with open(output_path, "w") as f:
        json.dump(coco, f, indent=2)

    print(f"COCO dataset saved to '{output_path}'")


# ==== MAIN ====
if __name__ == "__main__":
    # Rename columns for COCO compatibility
    synthetic_df = synthetic_df.rename(columns={
        "synthetic_mask_id": "image_id",
        "center_x": "ellipse_center_x",
        "center_y": "ellipse_center_y"
    })

    # Define output file path
    output_dir = os.path.join(path, "synthetic", "annotation")
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "synthetic.json")

    # Run conversion
    synthetic_df_to_coco_json(synthetic_df, output_path, image_width=256, image_height=256)


COCO dataset saved to '/Users/nuohaoliu/Documents/data_local/dataset/10/10_rot_flip_10k/all/synthetic/annotation/synthetic.json'
